# Entity extraction experiments

In [23]:
import json
import random
import sys

from dataclasses import asdict
from pathlib import Path

In [2]:
from gliner import GLiNER

In [3]:
sys.path.insert(0, '..')
from src.config import HF_TOKEN, LLM_MODEL_ID, DATA_PROCESSED_DIR, ENTITIES_PATH

In [4]:
from src.extraction.entity_extractor import EntityExtractor

## Trying GLiNER

In [5]:
model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")

/Users/nikhil.singh/PersonalProjects/kg-local-rag-gnn/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
chunks = [json.loads(l) for l in open(DATA_PROCESSED_DIR / 'chunks.jsonl')]

In [7]:
len(chunks)

188

In [8]:
labels = ["model", "author", "dataset", "task"]

In [9]:
for c in random.sample(chunks, 1):
    entities = model.predict_entities(c['text'], labels)

/Users/nikhil.singh/PersonalProjects/kg-local-rag-gnn/.venv/lib/python3.14/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 630 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


In [10]:
entities

[{'start': 0,
  'end': 6,
  'text': 'et al.',
  'label': 'author',
  'score': 0.6244040727615356},
 {'start': 18,
  'end': 26,
  'text': 'node2vec',
  'label': 'model',
  'score': 0.7259542942047119},
 {'start': 28,
  'end': 45,
  'text': 'Grover & Leskovec',
  'label': 'author',
  'score': 0.8506041765213013},
 {'start': 312,
  'end': 321,
  'text': 'Planetoid',
  'label': 'model',
  'score': 0.8511692881584167},
 {'start': 323,
  'end': 334,
  'text': 'Yang et al.',
  'label': 'author',
  'score': 0.9543233513832092},
 {'start': 532,
  'end': 543,
  'text': 'Gori et al.',
  'label': 'author',
  'score': 0.9819056987762451},
 {'start': 552,
  'end': 568,
  'text': 'Scarselli et al.',
  'label': 'author',
  'score': 0.9717566967010498},
 {'start': 805,
  'end': 814,
  'text': 'Li et al.',
  'label': 'author',
  'score': 0.9743034839630127},
 {'start': 940,
  'end': 955,
  'text': 'Duvenaud et al.',
  'label': 'author',
  'score': 0.9728906750679016},
 {'start': 1480,
  'end': 1496,
  '

In [11]:
for e in entities:
    print(e["label"], "->", e["text"])

author -> et al.
model -> node2vec
author -> Grover & Leskovec
model -> Planetoid
author -> Yang et al.
author -> Gori et al.
author -> Scarselli et al.
author -> Li et al.
author -> Duvenaud et al.
author -> Atwood & Towsley
author -> Niepert et al.
model -> conventional 1D convolutional neural network
author -> Bruna et al.
author -> Defferrard et al.


-------

## Trying a moderate size LLM

In [12]:
LLM_MODEL_ID

'meta-llama/Llama-3.3-70B-Instruct'

In [13]:
extractor = EntityExtractor()

In [14]:
chunks = [json.loads(l) for l in open(DATA_PROCESSED_DIR / 'chunks.jsonl')]

In [15]:
result = extractor.extract(chunks[0]['text'], chunk_id='test_0')

In [16]:
print(result)

ExtractionResult(entities=[Entity(type='Author', name='Bryan Perozzi', attributes={}), Entity(type='Author', name='Rami Al-Rfou', attributes={}), Entity(type='Author', name='Steven Skiena', attributes={}), Entity(type='Model', name='DeepWalk', attributes={}), Entity(type='Dataset', name='BlogCatalog', attributes={}), Entity(type='Dataset', name='Flickr', attributes={}), Entity(type='Dataset', name='YouTube', attributes={}), Entity(type='Paper', name="KDD'14", attributes={})], relationships=[Relationship(source='DeepWalk', relation='authored_by', target='Bryan Perozzi', attributes={}), Relationship(source='DeepWalk', relation='authored_by', target='Rami Al-Rfou', attributes={}), Relationship(source='DeepWalk', relation='authored_by', target='Steven Skiena', attributes={}), Relationship(source='DeepWalk', relation='evaluates_on', target='BlogCatalog', attributes={}), Relationship(source='DeepWalk', relation='evaluates_on', target='Flickr', attributes={}), Relationship(source='DeepWalk', 

### Random sample
----

In [17]:
for c in random.sample(chunks, 1):
    result = extractor.extract(c['text'], chunk_id='test_123')

In [18]:
result

ExtractionResult(entities=[Entity(type='Model', name='ChebNet', attributes={}), Entity(type='Model', name='CayleyNet', attributes={}), Entity(type='Model', name='Graph Convolutional Network', attributes={}), Entity(type='Model', name='GCN', attributes={}), Entity(type='Model', name='Adaptive Graph Convolutional Network', attributes={}), Entity(type='Model', name='AGCN', attributes={}), Entity(type='Model', name='Dual Graph Convolutional Network', attributes={}), Entity(type='Model', name='DGCN', attributes={}), Entity(type='Model', name='Spectral CNN', attributes={})], relationships=[Relationship(source='ChebNet', relation='improves_on', target='Spectral CNN', attributes={}), Relationship(source='CayleyNet', relation='improves_on', target='ChebNet', attributes={}), Relationship(source='Graph Convolutional Network', relation='evaluates_on', target='ChebNet', attributes={}), Relationship(source='GCN', relation='improves_on', target='ChebNet', attributes={}), Relationship(source='AGCN', r

------
## Running extraction using LLM on all chunks

In [25]:
texts = [(c['text'], f"{c['paper_id']}_chunk_{c['chunk_index']}") for c in chunks]
results = extractor.extract_batch(texts, batch_size=3, resume_from=ENTITIES_PATH)

In [26]:
len(results)

188

----

In [27]:
with open(ENTITIES_PATH, 'w') as f:
    for r in results:
        f.write(json.dumps(asdict(r)) + '\n')

# Script complete